# Classification NBA Model

## Configuration

## Imports

In [1]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    confusion_matrix,
    f1_score,
    make_scorer,
    precision_score,
    recall_score,
)
from nba_ou.data_preparation.missing_data.clean_df_for_training import (
    clean_dataframe_for_training
)
from sklearn.model_selection import cross_validate, train_test_split
from xgboost import XGBClassifier



## Load Data

In [3]:
data_path = "/home/adrian_alvarez/Projects/NBA_over_under_predictor/data/train_data/"
name = "all_odds_training_data_until_20260313.csv"

path = data_path + name

df_stats = pd.read_csv(path)

dtype_dict = {col: str for col in df_stats.columns if "ID" in col.upper()}

df_stats = pd.read_csv(
    path,
    dtype=dtype_dict
)


In [4]:
df_stats

,TOTAL_LINE_bet365,TEAM_ID_TEAM_HOME,TEAM_CITY_TEAM_HOME,TEAM_ABBREVIATION_TEAM_HOME,TEAM_NAME_TEAM_HOME,MATCHUP_TEAM_HOME,GAME_NUMBER_TEAM_HOME,TEAM_ID_TEAM_AWAY,TEAM_CITY_TEAM_AWAY,TEAM_ABBREVIATION_TEAM_AWAY,...,TRAVEL_RECENCY_RATIO_AWAY_2D_OVER_14D_BEFORE,REST_DAYS_DIFF_HOME_MINUS_AWAY_BEFORE,INJURY_PTS_SHARE_HOME_BEFORE,INJURY_PTS_SHARE_AWAY_BEFORE,STAR_PTS_PCT_DIFF_HOME_MINUS_AWAY_BEFORE,POSS_X_TSPCT_HOME_BEFORE,POSS_X_TSPCT_AWAY_BEFORE,IS_WEEKEND_BEFORE,MONTH_BEFORE,IS_US_HOLIDAY_BEFORE
0,NaN,1610612739,Cleveland,CLE,Cleveland Cavaliers,CLE vs. BOS,1,1610612738,Boston,BOS,...,NaN,0,NaN,NaN,NaN,NaN,NaN,0,10,0
1,NaN,1610612744,Golden State,GSW,Golden State Warriors,GSW vs. HOU,1,1610612745,Houston,HOU,...,NaN,0,NaN,NaN,NaN,NaN,NaN,0,10,0
2,NaN,1610612765,Detroit,DET,Detroit Pistons,DET vs. CHA,1,1610612766,Charlotte,CHA,...,NaN,0,NaN,NaN,NaN,NaN,NaN,0,10,0
3,NaN,1610612754,Indiana,IND,Indiana Pacers,IND vs. BKN,1,1610612751,Brooklyn,BKN,...,NaN,0,NaN,NaN,NaN,NaN,NaN,0,10,0
4,NaN,1610612753,Orlando,ORL,Orlando Magic,ORL vs. MIA,1,1610612748,Miami,MIA,...,NaN,0,NaN,NaN,NaN,NaN,NaN,0,10,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
11214,230.0,1610612745,Houston,HOU,Houston Rockets,HOU vs. NOP,66,1610612740,New Orleans,NOP,...,0.000000,0,0.275462,0.126165,0.022023,55.775104,56.532566,0,3,0
11215,236.5,1610612742,Dallas,DAL,Dallas Mavericks,DAL vs. CLE,67,1610612739,Cleveland,CLE,...,0.898440,-1,0.264739,0.286183,-0.038959,57.480455,59.474562,0,3,0
11216,224.0,1610612744,Golden State,GSW,Golden State Warriors,GSW vs. MIN,66,1610612750,Minnesota,MIN,...,0.000000,1,0.660259,0.025129,-0.101783,59.083429,61.478432,0,3,0
11217,235.0,1610612757,Portland,POR,Portland Trail Blazers,POR vs. UTA,67,1610612762,Utah,UTA,...,0.000000,1,0.358176,1.012906,0.038721,57.868186,57.539881,0,3,0


In [8]:
df_to_train = clean_dataframe_for_training(df_stats, nan_threshold=50,corr_threshold=0.982, max_na_per_row=7, create_missing_flags=False, verbose=1, keep_columns=['GAME_DATE'])

STARTING DATAFRAME CLEANING PIPELINE
Starting basic cleaning with 11219 rows
Basic cleaning complete: 8610 rows remaining

Starting advanced column cleaning with 3240 columns

Advanced column cleaning complete: 3240 → 1986 columns (1254 removed)


Dropping NA rows for SEASON_YEAR 2017...
   Removed 0 rows with NaN values from 2017 season

Applying missing data policy...

Missing Data Policy Report:
  Rows dropped: 0 (0.0%)
  Critical columns requiring data: 4
  Columns zero-filled: 112
  Infer pairs applied: 12/118
  Remaining NaN cells: 893340

Dropping rows with more than 7 NaN values...
Removed 3777 rows exceeding NaN threshold
CLEANING COMPLETE
Final shape: (4833, 1986)


In [9]:
# Count NAs per column
na_counts = df_to_train.isna().sum()

# Get most common SEASON_YEAR for nulls in each column
most_common_season = []
for col in df_to_train.columns:
    if na_counts[col] > 0:
        # Get rows where this column is null
        null_rows = df_to_train[df_to_train[col].isna()]
        if len(null_rows) > 0 and 'SEASON_YEAR' in df_to_train.columns:
            # Find most common SEASON_YEAR for these null rows
            common_season = null_rows['SEASON_YEAR'].mode()
            most_common_season.append(common_season.iloc[0] if len(common_season) > 0 else None)
        else:
            most_common_season.append(None)
    else:
        most_common_season.append(None)

na_counts_df = pd.DataFrame({
    'Column': na_counts.index,
    'NA_Count': na_counts.values,
    'NA_Percentage': (na_counts.values / len(df_to_train) * 100).round(2),
    'Most_Common_Season_Year': most_common_season
}).sort_values('NA_Count', ascending=False)

# Show only columns with NAs
na_counts_df[na_counts_df['NA_Count'] > 0]

,Column,NA_Count,NA_Percentage,Most_Common_Season_Year
1798,total_pct_money_under,147,3.04,2021.0
1797,total_pct_bets_under,147,3.04,2021.0
1802,moneyline_pct_money_home,144,2.98,2021.0
1801,moneyline_pct_bets_home,144,2.98,2021.0
1800,spread_pct_money_home,142,2.94,2021.0
1799,spread_pct_bets_home,142,2.94,2021.0
1803,total_consensus_pct_under,73,1.51,2024.0
1804,spread_consensus_pct_home,69,1.43,2024.0
1664,total_consensus_pct_under_TREND_SLOPE_LAST_5_H...,17,0.35,2023.0
1668,spread_consensus_pct_home_TREND_SLOPE_LAST_5_H...,17,0.35,2023.0


In [10]:
BET365_LINE_COL =  "TOTAL_LINE_bet365"

# Ensure scoring line and target exist (avoid NaN-driven undefined betting accuracy).
df_to_train = df_to_train.dropna(subset=[BET365_LINE_COL, "TOTAL_POINTS"]).copy()
df_to_train['GAME_DATE'] = pd.to_datetime(df_to_train['GAME_DATE'])
df_to_train = df_to_train.sort_values("GAME_DATE").reset_index(drop=True)

In [11]:
print(f" Total Seasons in dataset: {df_to_train['SEASON_YEAR'].nunique()}")
print(f" Total Games in dataset: {len(df_to_train)}")
print(f"Seasons: {df_to_train['SEASON_YEAR'].unique()}")
#print number of games per season
print(df_to_train['SEASON_YEAR'].value_counts())

 Total Seasons in dataset: 5
 Total Games in dataset: 4833
Seasons: [2021 2022 2023 2024 2025]
SEASON_YEAR
2024    1248
2021    1241
2022    1232
2025     927
2023     185
Name: count, dtype: int64


In [12]:
df_to_train = df_to_train[df_to_train['SEASON_YEAR'] >= 2023].copy()

In [10]:
df_to_train = df_to_train[df_to_train['SEASON_YEAR'] >= 2024].copy()

## Train / Test

In [11]:
# X = df_to_train.drop(["TOTAL_POINTS",  "GAME_DATE"], axis=1, errors="ignore")
X = df_to_train.drop(["TOTAL_POINTS", "SEASON_YEAR", "GAME_DATE"], axis=1, errors="ignore")

y = pd.to_numeric(df_to_train["TOTAL_POINTS"], errors="coerce")


In [12]:
# Train / Test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.075, random_state=16, shuffle=False)

In [35]:
print(f"Training set size: {X_train.shape[0]}")
print(f"Test set size: {X_test.shape[0]}")
# Check number of coulmns
print(f"Number of columns in training set: {X_train.shape[1]}")
print(f"Number of columns in test set: {X_test.shape[1]}")

Training set size: 2011
Test set size: 164
Number of columns in training set: 1983
Number of columns in test set: 1983


## Cross-validation

In [16]:
from sklearn.model_selection import TimeSeriesSplit, cross_validate, cross_val_predict
from sklearn.metrics import mean_squared_error, mean_absolute_error, make_scorer, root_mean_squared_error


In [17]:
# Declare time-series cross-validation strategy
kf = TimeSeriesSplit(n_splits=5)


In [18]:
import numpy as np


def over_under_betting_accuracy(
    true_total: np.ndarray,
    pred_total: np.ndarray,
    betting_line: np.ndarray,
) -> float:
    true_total = np.asarray(true_total, dtype=float)
    pred_total = np.asarray(pred_total, dtype=float)
    betting_line = np.asarray(betting_line, dtype=float)

    valid = np.isfinite(true_total) & np.isfinite(pred_total) & np.isfinite(betting_line)
    if not np.any(valid):
        return np.nan

    true_total = true_total[valid]
    pred_total = pred_total[valid]
    betting_line = betting_line[valid]

    true_side = np.where(
        true_total > betting_line,
        "OVER",
        np.where(true_total < betting_line, "UNDER", "PUSH"),
    )
    pred_side = np.where(
        pred_total > betting_line,
        "OVER",
        np.where(pred_total < betting_line, "UNDER", "PUSH"),
    )

    non_push = (true_side != "PUSH") & (pred_side != "PUSH")
    if not np.any(non_push):
        return np.nan

    return float(np.mean(true_side[non_push] == pred_side[non_push]))


class OverUnderScorer:
    """Custom sklearn-compatible scorer for over/under betting accuracy using bet365 line."""

    def __init__(self, line_col: str):
        self.line_col = line_col

    def __call__(self, estimator, X, y_true):
        if self.line_col not in X.columns:
            raise KeyError(f"{self.line_col} not found in X for scoring")

        y_pred = estimator.predict(X)
        betting_line = pd.to_numeric(X[self.line_col], errors="coerce").to_numpy(dtype=float)

        return over_under_betting_accuracy(
            true_total=np.asarray(y_true, dtype=float),
            pred_total=np.asarray(y_pred, dtype=float),
            betting_line=betting_line,
        )


over_under_scorer = OverUnderScorer(BET365_LINE_COL)


In [19]:
# Declare scores to be used
scoring = {
    'MSE': make_scorer(mean_squared_error),
    'RMSE': make_scorer(root_mean_squared_error),
    'MAE': make_scorer(mean_absolute_error),
    'OU_Betting_Accuracy': over_under_scorer,
}

In [20]:
def print_metrics(cv_results):
    for sc in scoring.keys():
        if sc == 'OU_Betting_Accuracy':
            print(f'Train {sc}:', f"{cv_results[f'train_{sc}'].mean():.2%}")
            print(f'Validation {sc}:', f"{cv_results[f'test_{sc}'].mean():.2%}")
        else:
            print(f'Train {sc}:', cv_results[f'train_{sc}'].mean().round(5))
            print(f'Validation {sc}:', cv_results[f'test_{sc}'].mean().round(5))
        print()

In [21]:
def real_vs_pred(model, X_train, y_train):
    preds = cross_val_predict(model, X_train, y_train, cv=kf, n_jobs=-1)
    x_line = np.arange(y_train.min(), y_train.max())
    plt.scatter(y_train, preds)
    plt.plot(x_line, x_line, color='orange')
    plt.xlabel('Real target')
    plt.ylabel('Predicted target')
    plt.show()

## Baseline

In [22]:
from sklearn.dummy import DummyRegressor

season_bl = DummyRegressor(strategy='mean')
cv_results = cross_validate(season_bl, X_train, y_train, cv=kf,
                            scoring=scoring, return_train_score=True)
season_bl.fit(X_train, y_train)
print_metrics(cv_results)

Train MSE: 388.67677
Validation MSE: 379.45211

Train RMSE: 19.71472
Validation RMSE: 19.47193

Train MAE: 15.99103
Validation MAE: 15.70488

Train OU_Betting_Accuracy: 49.37%
Validation OU_Betting_Accuracy: 48.23%



In [ ]:
# Baseline 3: Predict the opening betting line as total points
y_pred_baseline_3 = pd.to_numeric(X_train['TOTAL_LINE_consensus_opener'], errors="coerce")
valid = y_pred_baseline_3.notna() & pd.to_numeric(y_train, errors="coerce").notna()

# Evaluate
mse = mean_squared_error(y_train[valid], y_pred_baseline_3[valid])
mae = mean_absolute_error(y_train[valid], y_pred_baseline_3[valid])
rmse = root_mean_squared_error(y_train[valid], y_pred_baseline_3[valid])
OU_accuracy = over_under_betting_accuracy(
    true_total=y_train[valid],
    pred_total=y_pred_baseline_3[valid],
    betting_line=X_train.loc[valid, 'TOTAL_LINE_bet365']
)
print(f"MSE: {mse:.2f}")
print(f"RMSE: {rmse:.2f}")
print(f"MAE: {mae:.2f}")
print(f"Over/Under Accuracy: {OU_accuracy:.2f}")    

MSE: 307.64
RMSE: 17.54
MAE: 14.10
Over/Under Accuracy: 0.48


## Logistic Regression

In [36]:
from sklearn.linear_model import LinearRegression

lr = LinearRegression()
cv_results = cross_validate(lr, X_train.fillna(0), y_train, cv=kf,
                            scoring=scoring, return_train_score=True)

print_metrics(cv_results)

Train MSE: 11.82486
Validation MSE: 2224456278.41482

Train RMSE: 1.94804
Validation RMSE: 21178.91834

Train MAE: 1.4954
Validation MAE: 9642.08216

Train OU_Betting_Accuracy: 96.64%
Validation OU_Betting_Accuracy: 50.32%



In [37]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from xgboost import XGBRegressor

# Example XGBoost regressor:
xgb_reg = XGBRegressor(
    max_depth=4,
    learning_rate=0.02,
    n_estimators=350,
    subsample=0.5,       # Equivalent to max_samples in GBRegressor
    colsample_bytree=0.6, # Equivalent to max_features in GBRegressor
    n_jobs=-2,
    random_state=16)

cv_results = cross_validate(
    xgb_reg, 
    X_train, y_train, 
    cv=kf, 
    scoring=scoring,      # Use your custom scoring or e.g. 'neg_mean_absolute_error'
    return_train_score=True,
    n_jobs=-2
)

# Print metrics
print_metrics(cv_results)
# Train final model on full train set
xgb_reg.fit(X_train, y_train)

Train MSE: 53.14203
Validation MSE: 306.92037

Train RMSE: 6.95295
Validation RMSE: 17.51603

Train MAE: 5.55628
Validation MAE: 14.08436

Train OU_Betting_Accuracy: 93.87%
Validation OU_Betting_Accuracy: 47.20%



XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=0.6, device=None, early_stopping_rounds=None,
             enable_categorical=False, eval_metric=None, feature_types=None,
             gamma=None, grow_policy=None, importance_type=None,
             interaction_constraints=None, learning_rate=0.02, max_bin=None,
             max_cat_threshold=None, max_cat_to_onehot=None,
             max_delta_step=None, max_depth=4, max_leaves=None,
             min_child_weight=None, missing=nan, monotone_constraints=None,
             multi_strategy=None, n_estimators=350, n_jobs=-2,
             num_parallel_tree=None, random_state=16, ...)

In [38]:
feature_importances = xgb_reg.feature_importances_
important_features = np.argsort(feature_importances)[::-1]  
feature_importances = pd.DataFrame({
    'Feature': X_train.columns[important_features],
    'Importance': feature_importances[important_features]
}).sort_values(by="Importance", ascending=False)
feature_importances



,Feature,Importance
0,TOTAL_LINE_consensus_opener,0.006707
1,TOTAL_LINE_caesars,0.004279
2,TOTAL_LINE_draftkings,0.003702
3,TOTAL_LINE_betmgm,0.002902
4,TOTAL_LINE_bet365,0.002363
...,...,...
1667,DIFF_FROM_LINE_betmgm_LAST_ALL_10_MATCHES_BEFO...,0.000000
1668,DIFF_FROM_LINE_betmgm_SEASON_BEFORE_AVG_TEAM_HOME,0.000000
1669,total_consensus_pct_under_LAST_HOME_AWAY_5_MAT...,0.000000
1670,total_consensus_pct_over_LAST_ALL_10_MATCHES_B...,0.000000


In [39]:
import numpy as np
import pandas as pd


def evaluate_ou_thresholds(
    model,
    X_test: pd.DataFrame,
    y_test_total: pd.Series,
    line_col: str,
    thresholds=(0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10),
):
    """Evaluate OU accuracy at different confidence thresholds using bet365 line."""
    if line_col not in X_test.columns:
        raise KeyError(f"{line_col} not found in X_test")

    y_pred_total = np.asarray(model.predict(X_test), dtype=float)
    y_true_total = pd.to_numeric(y_test_total, errors="coerce").to_numpy(dtype=float)
    line = pd.to_numeric(X_test[line_col], errors="coerce").to_numpy(dtype=float)

    margin = np.abs(y_pred_total - line)

    rows = []
    n_total = len(y_true_total)

    for t in thresholds:
        mask = margin > t
        n = int(mask.sum())

        if n == 0:
            acc = np.nan
        else:
            acc = over_under_betting_accuracy(
                true_total=y_true_total[mask],
                pred_total=y_pred_total[mask],
                betting_line=line[mask],
            )

        rows.append(
            {
                "threshold_abs_pred_vs_line_gt": t,
                "n_games": n,
                "pct_of_test": (n / n_total) if n_total else np.nan,
                "ou_accuracy": acc,
            }
        )

    return pd.DataFrame(rows), y_pred_total


results_df, y_pred_test_total = evaluate_ou_thresholds(
    model=xgb_reg,
    X_test=X_test,
    y_test_total=y_test,
    line_col=BET365_LINE_COL,
    thresholds=range(0, 11),
)

display(
    results_df.style.format(
        {"pct_of_test": "{:.1%}", "ou_accuracy": "{:.2%}"}
    )
)


,threshold_abs_pred_vs_line_gt,n_games,pct_of_test,ou_accuracy
0,0,164,100.0%,54.66%
1,1,116,70.7%,54.39%
2,2,81,49.4%,55.70%
3,3,47,28.7%,58.70%
4,4,28,17.1%,57.14%
5,5,13,7.9%,69.23%
6,6,5,3.0%,60.00%
7,7,2,1.2%,50.00%
8,8,1,0.6%,0.00%
9,9,1,0.6%,0.00%


In [29]:
X = df_to_train.drop(["TOTAL_POINTS",  "GAME_DATE"], axis=1, errors="ignore")
# X = df_to_train.drop(["TOTAL_POINTS", "SEASON_YEAR", "GAME_DATE"], axis=1, errors="ignore")

y = pd.to_numeric(df_to_train["TOTAL_POINTS"], errors="coerce")

# Train / Test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.025, random_state=16, shuffle=False)
len(X_train), len(X_test)

(4679, 120)

In [40]:
from nba_ou.prediction.prediction_tabpfn_client import _init_tabpfn_client
import pandas as pd
from sklearn.metrics import mean_absolute_error, mean_squared_error, root_mean_squared_error

TabPFNRegressor = _init_tabpfn_client()
tabpfn_reg = TabPFNRegressor()

tabpfn_reg.fit(X_train, y_train, )
y_pred_test_tabpfn = pd.Series(tabpfn_reg.predict(X_test), index=X_test.index, name="PRED_TOTAL_POINTS_TABPFN")

valid_test = y_test.notna() & y_pred_test_tabpfn.notna()
print(f"TabPFN Test MSE: {mean_squared_error(y_test[valid_test], y_pred_test_tabpfn[valid_test]):.4f}")
print(f"TabPFN Test MAE: {mean_absolute_error(y_test[valid_test], y_pred_test_tabpfn[valid_test]):.4f}")
print(f"TabPFN Test RMSE: {root_mean_squared_error(y_test[valid_test], y_pred_test_tabpfn[valid_test]):.4f}")


  Welcome Back! Found existing access token, reusing it for authentication.


Processing: 100%|██████████| [00:09<00:00]

TabPFN Test MSE: 348.9971
TabPFN Test MAE: 14.5651
TabPFN Test RMSE: 18.6815


In [41]:
results_tabpfn_df, _ = evaluate_ou_thresholds(
    model=tabpfn_reg,
    X_test=X_test,
    y_test_total=y_test,
    line_col=BET365_LINE_COL,
    thresholds=np.arange(0, 5, 0.5),
)

display(
    results_tabpfn_df.style.format(
        {"pct_of_test": "{:.1%}", "ou_accuracy": "{:.2%}"}
    )
)

Processing: 100%|██████████| [00:09<00:00]


,threshold_abs_pred_vs_line_gt,n_games,pct_of_test,ou_accuracy
0,0.000000,164,100.0%,53.42%
1,0.500000,148,90.2%,55.17%
2,1.000000,115,70.1%,53.98%
3,1.500000,84,51.2%,54.76%
4,2.000000,57,34.8%,59.65%
5,2.500000,46,28.0%,63.04%
6,3.000000,30,18.3%,56.67%
7,3.500000,15,9.1%,53.33%
8,4.000000,6,3.7%,50.00%
9,4.500000,3,1.8%,33.33%


# with no year

In [25]:
X = df_to_train.drop(["TOTAL_POINTS", "SEASON_YEAR", "GAME_DATE"], axis=1, errors="ignore")

y = pd.to_numeric(df_to_train["TOTAL_POINTS"], errors="coerce")

# Train / Test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.075, random_state=16, shuffle=False)

In [ ]:
from nba_ou.prediction.prediction_tabpfn_client import _init_tabpfn_client
import pandas as pd
from sklearn.metrics import mean_absolute_error, mean_squared_error, root_mean_squared_error

TabPFNRegressor = _init_tabpfn_client()
tabpfn_reg = TabPFNRegressor()

tabpfn_reg.fit(X_train, y_train, )
y_pred_test_tabpfn = pd.Series(tabpfn_reg.predict(X_test), index=X_test.index, name="PRED_TOTAL_POINTS_TABPFN")

valid_test = y_test.notna() & y_pred_test_tabpfn.notna()
print(f"TabPFN Test MSE: {mean_squared_error(y_test[valid_test], y_pred_test_tabpfn[valid_test]):.4f}")
print(f"TabPFN Test MAE: {mean_absolute_error(y_test[valid_test], y_pred_test_tabpfn[valid_test]):.4f}")
print(f"TabPFN Test RMSE: {root_mean_squared_error(y_test[valid_test], y_pred_test_tabpfn[valid_test]):.4f}")

results_tabpfn_df, _ = evaluate_ou_thresholds(
    model=tabpfn_reg,
    X_test=X_test,
    y_test_total=y_test,
    line_col=BET365_LINE_COL,
    thresholds=np.arange(0, 5, 0.5),
)

display(
    results_tabpfn_df.style.format(
        {"pct_of_test": "{:.1%}", "ou_accuracy": "{:.2%}"}
    )
)

Processing: 100%|██████████| [00:13<00:00]


TabPFN Test MSE: 299.2075
TabPFN Test MAE: 13.6329
TabPFN Test RMSE: 17.2976


Processing: 100%|██████████| [00:13<00:00]


,threshold_abs_pred_vs_line_gt,n_games,pct_of_test,ou_accuracy
0,0,360,100.0%,54.83%
1,1,193,53.6%,49.21%
2,2,88,24.4%,50.00%
3,3,17,4.7%,47.06%
4,4,4,1.1%,50.00%
5,5,3,0.8%,33.33%
6,6,1,0.3%,100.00%
7,7,1,0.3%,100.00%
8,8,1,0.3%,100.00%
9,9,1,0.3%,100.00%
